# Data Augmentation Tutorial for Deployment on Picsellia

This tutorial will guide you through creating and deploying a data augmentation **processing pipeline** on Picsellia. The steps include:
1. Implementing a custom data augmentation function (`apply_augmentations`) with a specific input-output structure.
2. Deploying this function as a Picsellia processing pipeline with minimal setup.

---

## Key Steps

### Step 1: Define Requirements

The pipeline depends on specific Python packages, which should be listed in a `requirements.txt` file located in the same directory as this notebook.

#### Default Requirements:
```plaintext
pillow==9.0.0
albumentations==1.3.0
opencv-python-headless==4.5.5.64
picsellia==6.18.3
```

#### Adding More Requirements
If your `apply_augmentations` function uses additional libraries, such as PyTorch or TensorFlow, add them to the requirements.txt file.

For example:
```plaintext
torch==1.12.0
```

After updating the requirements file, re-run the deployment step to ensure the new dependencies are included.

### Step 2: Implement apply_augmentations
Edit the function `apply_augmentations` in the file `utils/custom_augmentations.py`.

**Function Signature:**
```python
def apply_augmentations(img: Image.Image, annotations: List[Dict]) -> Tuple[List[Image.Image], List[List[Dict]]]
```

#### Input Format
- `img`: A `PIL.Image.Image` object representing the input image.
- `annotations`: A list of dictionaries in COCO format, each dictionary containing:
    - `id`: Unique identifier for the annotation.
    - `image_id`: Identifier for the image this annotation belongs to.
    - `category_id`: Identifier for the category of the object.
    - `bbox`: Bounding box in the format `[x_min, y_min, width, height]`.
    - `iscrowd`: Indicates if the annotation is a crowd region (default: `0`).
    - `segmentation`: List of polygons describing the segmentation mask (optional, default: `[]`).
    - `area`: Area of the bounding box (optional, default: `0.0`).
    - `score`: Confidence score (optional, default: `0.0`).

#### Output Format
- `augmented_images`: A list of `PIL.Image.Image` objects representing the augmented images.
- `augmented_annotations`: A list of lists, where each inner list contains dictionaries in the COCO format, updated to reflect the augmentations.

### Step 3: Test Locally
You can test your augmentation logic using sample images and annotations provided in this notebook. Ensure that:

- Bounding boxes in the annotations are correctly updated for transformations like flips or rotations.
- The format of the annotations matches the COCO specification.

### Step 4: Deploy on Picsellia
Use the provided deployment function `deploy_on_picsellia` to register your processing pipeline with Picsellia.


## Why this Approach?
By following this tutorial, you will:

- Focus only on the core augmentation logic.
- Avoid dealing with the complexities of pipeline setup, logging, and deployment manually.
- Seamlessly integrate your custom logic with Picsellia’s platform to process your dataset versions.

In [ ]:
from albumentations import Compose, BboxParams

def get_augmentation_pipeline() -> Compose:
    """
    Define and return the augmentation pipeline.

    Returns:
        Compose: An Albumentations Compose object with defined augmentations.
    """
    return Compose(
        [
            RandomBrightnessContrast(p=0.5),
            HorizontalFlip(p=0.5),
            ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7),
            Blur(blur_limit=3, p=0.3),
        ],
        bbox_params=BboxParams(format="coco", label_fields=["id"]),
    )

In [11]:
from examples.processing.augmentation.utils.custom_augmentations import build_annotation, calculate_area
from typing import List, Dict, Tuple
from PIL import Image
import numpy as np
from albumentations import Compose, RandomBrightnessContrast, HorizontalFlip, ShiftScaleRotate, Blur

def apply_augmentations(
    img: Image.Image, annotations: List[Dict]
) -> Tuple[List[Image.Image], List[List[Dict]]]:
    """
    Generate multiple augmented versions of the input image, updating annotations accordingly.

    Args:
        img (PIL.Image.Image): Input image.
        annotations (List[Dict]): List of COCO annotations for the image.

    Returns:
        Tuple[List[Image.Image], List[List[Dict]]]: List of augmented images and their corresponding annotations.
    """
    # Convert PIL Image to NumPy array for Albumentations
    img_array = np.array(img)

    # Map annotations by ID for easy retrieval
    annotations_by_id = {ann["id"]: ann for ann in annotations}

    # Fetch the augmentation pipeline
    augmentation_pipeline = get_augmentation_pipeline()

    # Generate multiple augmented images and annotations
    augmented_images = []
    augmented_annotations = []

    for _ in range(3):  # Generate 3 augmentations
        augmented = augmentation_pipeline(
            image=img_array,
            bboxes=[ann["bbox"] for ann in annotations],
            id=[ann["id"] for ann in annotations],
        )

        # Convert augmented image back to PIL format
        augmented_img = Image.fromarray(augmented["image"])

        # Rebuild annotations with updated bounding boxes
        updated_annotations = [
            build_annotation(
                annotations_by_id[obj_id],
                bbox,
                calculate_area(bbox),
            )
            for bbox, obj_id in zip(augmented["bboxes"], augmented["id"])
        ]

        # Append results
        augmented_images.append(augmented_img)
        augmented_annotations.append(updated_annotations)

    return augmented_images, augmented_annotations

## Deploying on Picsellia

Once you’ve defined your custom `apply_augmentations` function and tested it, deploy it to Picsellia by calling the `setup_dockerized_pipeline` function below.


In [19]:
from src.models.utils.processing_creation import setup_dockerized_pipeline

# Example Deployment
setup_dockerized_pipeline(
    api_token="",
    organization_id="",
    processing_name="augmentations_pipeline",
    pipeline_script_path="augmentations_pipeline.py",
    requirements_file_path="requirements.txt",
    docker_image="soniagrh/processing-augmentations",
    docker_tag="latest",
    default_parameters={
        "augmentation_probability": 0.5,
        "datalake": "default",
        "data_tag": "augmented_data",
    },
)

Dockerized pipeline 'augmentations_pipeline' created in /home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/src/pipelines/augmentations_pipeline/
You are using an outdated version of the picsellia package (6.18.2)
Please consider upgrading to 6.18.3 with pip install picsellia --upgrade
Hi SoniaGrh, welcome back. 🥑
Workspace: SoniaGrh's organization.
